In [1]:
import cv2 as cv
import numpy as np
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


def load_rgb(image_path):
    image_bgr = cv.imread(image_path)
    return cv.cvtColor(image_bgr, cv.COLOR_BGR2RGB)


def find_circles(image, *, blur, min_dist, param1, param2, min_radius, max_radius):
    gray = cv.cvtColor(image, cv.COLOR_RGB2GRAY)
    if blur > 1:
        gray = cv.medianBlur(gray, blur)

    circles = cv.HoughCircles(
        gray,
        cv.HOUGH_GRADIENT,
        dp=1,
        minDist=min_dist,
        param1=param1,
        param2=param2,
        minRadius=min_radius,
        maxRadius=max_radius,
    )
    return np.round(circles[0]).astype(int)


def draw_circles(image, circles, offset=(0, 0)):
    annotated = image.copy()
    x_offset, y_offset = offset

    for x, y, radius in circles:
        center = (x + x_offset, y + y_offset)
        cv.circle(annotated, center, radius, (0, 255, 0), 2)
        cv.circle(annotated, center, 2, (255, 0, 0), 3)

    return annotated, len(circles)


def show_result(image, label, count, source_path):
    result_path = RESULTS_DIR / f"{Path(source_path).stem}_result.jpg"
    image_bgr = cv.cvtColor(image, cv.COLOR_RGB2BGR)
    cv.imwrite(
        str(result_path),
        image_bgr,
        [cv.IMWRITE_JPEG_QUALITY, 92],
    )
    print(f"{label}: {count}")
    print(f"Saved: {result_path}")


def run_hough(image_path, *, params, roi=None, label='Total circles'):
    image = load_rgb(image_path)
    if roi is None:
        crop = image
        offset = (0, 0)
    else:
        y1, y2, x1, x2 = roi
        crop = image[y1:y2, x1:x2]
        offset = (x1, y1)

    circles = find_circles(crop, **params)
    annotated, count = draw_circles(image, circles, offset)
    show_result(annotated, label, count, image_path)
    return annotated, count


def run_wood_logs(
    image_path,
    *,
    params,
    stack_top_x=45,
    lower_wood=(5, 25, 65),
    upper_wood=(32, 220, 255),
    min_wood_ratio=0.45,
):
    image = load_rgb(image_path)
    height, width = image.shape[:2]
    circles = find_circles(image, **params)

    stack_mask = np.zeros((height, width), dtype=np.uint8)
    stack_polygon = np.array(
        [[0, 0], [stack_top_x, 0], [width, height], [0, height]],
        dtype=np.int32,
    )
    cv.fillPoly(stack_mask, [stack_polygon], 255)

    hsv = cv.cvtColor(image, cv.COLOR_RGB2HSV)
    wood_mask = cv.inRange(
        hsv,
        np.array(lower_wood, dtype=np.uint8),
        np.array(upper_wood, dtype=np.uint8),
    )

    filtered = []
    for x, y, radius in circles:
        if not (0 <= x < width and 0 <= y < height):
            continue
        if stack_mask[y, x] == 0:
            continue

        circle_mask = np.zeros((height, width), dtype=np.uint8)
        cv.circle(circle_mask, (x, y), max(1, int(radius * 0.6)), 255, -1)

        wood_pixels = cv.countNonZero(cv.bitwise_and(wood_mask, circle_mask))
        total_pixels = cv.countNonZero(circle_mask)
        if wood_pixels / total_pixels >= min_wood_ratio:
            filtered.append((x, y, radius))

    annotated, count = draw_circles(image, filtered)
    show_result(annotated, 'Total logs', count, image_path)
    return annotated, count


## Брёвна

In [2]:
logs1_result, logs1_count = run_hough(
    "logs-1.jpg",
    params={
        "blur": 5,
        "min_dist": 50,
        "param1": 130,
        "param2": 26,
        "min_radius": 12,
        "max_radius": 70,
    },
    roi=(10, 800, 300, 1020),
    label="Total logs",
)

Total logs: 60
Saved: results/logs-1_result.jpg


In [3]:
logs2_result, logs2_count = run_hough(
    "logs-2.jpg",
    params={
        "blur": 7,
        "min_dist": 45,
        "param1": 110,
        "param2": 24,
        "min_radius": 14,
        "max_radius": 70,
    },
    roi=(10, 800, 170, 870),
    label="Total logs",
)

Total logs: 60
Saved: results/logs-2_result.jpg


In [4]:
logs3_result, logs3_count = run_hough(
    "logs-3.jpg",
    params={
        "blur": 5,
        "min_dist": 25,
        "param1": 110,
        "param2": 28,
        "min_radius": 4,
        "max_radius": 53,
    },
    roi=(900, 1420, 350, 800),
    label="Total logs",
)

Total logs: 59
Saved: results/logs-3_result.jpg


In [5]:
logs4_result, logs4_count = run_wood_logs(
    "logs-4.jpg",
    params={
        "blur": 3,
        "min_dist": 24,
        "param1": 100,
        "param2": 25,
        "min_radius": 7,
        "max_radius": 35,
    },
    stack_top_x=45,
)

Total logs: 76
Saved: results/logs-4_result.jpg


## Трубы


In [6]:
pipes4_result, pipes4_count = run_hough(
    "pipes-4.jpg",
    params={
        "blur": 5,
        "min_dist": 11,
        "param1": 120,
        "param2": 24,
        "min_radius": 5,
        "max_radius": 25,
    },
    label="Total pipes",
)

Total pipes: 78
Saved: results/pipes-4_result.jpg


In [7]:
pipes5_result, pipes5_count = run_hough(
    "pipes-5.jpg",
    params={
        "blur": 3,
        "min_dist": 11,
        "param1": 120,
        "param2": 24,
        "min_radius": 4,
        "max_radius": 25,
    },
    label="Total pipes",
)

Total pipes: 133
Saved: results/pipes-5_result.jpg


In [8]:
pipes6_result, pipes6_count = run_hough(
    "pipes-6.jpg",
    params={
        "blur": 5,
        "min_dist": 22,
        "param1": 130,
        "param2": 36,
        "min_radius": 12,
        "max_radius": 48,
    },
    label="Total pipes",
)

Total pipes: 58
Saved: results/pipes-6_result.jpg
